# Generic Lab Report OCR — Step by Step

This notebook has two simple parts:

1. **Text OCR:** reads text, confidence scores, and coordinates.
2. **Document structure:** detects document blocks such as titles, text, and tables.

The code does not search for CBC-specific field names. You can use the same cells with another clear lab report by changing only `INPUT_PATH`.

In [1]:
# Cell 1: Import libraries and set file paths
from pathlib import Path
from io import StringIO
import json

import pandas as pd
from IPython.display import Markdown, display
from paddleocr import PaddleOCR, PPStructureV3

INPUT_PATH = Path("data/input/cbc_sample_report.pdf")
OCR_OUTPUT_FOLDER = Path("outputs/ocr")
STRUCTURE_OUTPUT_FOLDER = Path("outputs/structure")

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_PATH.resolve()}")

print("Input file:", INPUT_PATH.resolve())

Input file: C:\Users\user\Desktop\Paddle OCR\data\input\cbc_sample_report.pdf


## Part 1: Read all text with PaddleOCR

In [2]:
# Cell 2: Load the text detection and recognition models
ocr = PaddleOCR(
    device="cpu",
    text_detection_model_name="PP-OCRv6_small_det",
    text_recognition_model_name="PP-OCRv6_small_rec",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    enable_mkldnn=False,
)

print("OCR model loaded")

Creating model: ('PP-OCRv6_small_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\user\.paddlex\official_models\PP-OCRv6_small_det`.
C:\Users\user\anaconda3\envs\paddle_ocr\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv6_small_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\user\.paddlex\official_models\PP-OCRv6_small_rec`.


OCR model loaded


In [3]:
# Run OCR and keep only the useful values
paddle_results = list(ocr.predict(str(INPUT_PATH)))
ocr_data = []

for page_number, result in enumerate(paddle_results, start=1):
    page_data = result.json

    if "res" in page_data:
        page_data = page_data["res"]

    texts = page_data.get("rec_texts", [])
    scores = page_data.get("rec_scores", [])
    boxes = page_data.get("rec_boxes", [])

    for text, score, box in zip(texts, scores, boxes):
        if not str(text).strip() or len(box) != 4:
            continue

        x1, y1, x2, y2 = [int(value) for value in box]

        ocr_data.append({
            "page": page_number,
            "text": str(text).strip(),
            "confidence": round(float(score), 4),
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2,
        })

print("Pages read:", len(paddle_results))
print("Text boxes found:", len(ocr_data))

Pages read: 1
Text boxes found: 205


In [4]:
# Cell 4: View the OCR result as a table
ocr_table = pd.DataFrame(ocr_data)
ocr_table.head(25)

,page,text,confidence,x1,y1,x2,y2
0,1,THE COMPLETE BLOOD COUNT SAMPLE REPORT,0.9973,319,111,946,134
1,1,1.,0.9999,1253,118,1273,136
2,1,Name and address of the lab where the test was...,0.9853,1285,116,1822,138
3,1,"Tests may be run in a physician office lab, a ...",0.9903,1290,141,1816,163
4,1,Different laboratories generate reports that c...,0.9976,100,160,1134,184
5,1,"clinic or hospital, and/or samples may be sent...",0.9999,1288,163,1824,188
6,1,information included. This is one example of w...,0.9934,98,189,1153,214
7,1,laboratory for analysis.,0.9987,1290,190,1489,211
8,1,Names and places used have been made up for il...,0.9916,99,221,1149,245
9,1,2.,1.0000,1250,225,1276,246


In [5]:
# Put the recognized text in reading order
ordered_data = sorted(
    ocr_data,
    key=lambda item: (item["page"], item["y1"], item["x1"]),
)

recognized_text = "\n".join(item["text"] for item in ordered_data)
print(recognized_text)

THE COMPLETE BLOOD COUNT SAMPLE REPORT
Name and address of the lab where the test was performed.
1.
Tests may be run in a physician office lab, a lab located in a
Different laboratories generate reports that can vary greatly in appearance and in the order and kind of
clinic or hospital, and/or samples may be sent to a reference
information included. This is one example of what a lab report for a Complete Blood Count may look like.
laboratory for analysis.
Names and places used have been made up for illustrative purposes only. The numbered key to the right
2.
Date this copy of the report was printed. This date may be
different than the date the results were generated, especially
explains a few of the report elements.
on cumulative reports (those that include results of several
different tests run on different days).
3.
Patient name or identifier. Links results to the correct person.
University Medical Center, Dept. of Pathology
Report Date/Time:
2
Patient identifier and identification n

In [6]:
# Find text that may need human review
CONFIDENCE_LIMIT = 0.90

low_confidence = ocr_table[
    ocr_table["confidence"] < CONFIDENCE_LIMIT
]

print("Low-confidence text boxes:", len(low_confidence))
low_confidence

Low-confidence text boxes: 3


,page,text,confidence,x1,y1,x2,y2
26,1,３４5,0.5931,64,417,93,515
46,1,e日,0.3699,1008,507,1071,611
100,1,L**,0.8322,772,840,813,868


In [7]:
# Save the text OCR outputs
OCR_OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

with (OCR_OUTPUT_FOLDER / "ocr_result.json").open("w", encoding="utf-8") as file:
    json.dump(ocr_data, file, indent=2, ensure_ascii=False)

ocr_table.to_csv(OCR_OUTPUT_FOLDER / "ocr_result.csv", index=False)
(OCR_OUTPUT_FOLDER / "recognized_text.txt").write_text(
    recognized_text,
    encoding="utf-8",
)

for result in paddle_results:
    result.save_to_img(str(OCR_OUTPUT_FOLDER))

print("OCR files saved in:", OCR_OUTPUT_FOLDER.resolve())

OCR files saved in: C:\Users\user\Desktop\Paddle OCR\outputs\ocr


## Part 2: Extract simple label and value pairs

Many reports write metadata as a label followed by a colon, such as `Name:` or `Report Date:`. The next cell finds these labels and selects the nearest text on the same line. It does not contain a list of medical field names.

In [8]:
# Find simple Label: Value metadata pairs
metadata_rows = []

for label_item in ocr_data:
    text = label_item["text"]

    if ":" not in text:
        continue

    label, value = text.split(":", maxsplit=1)
    label = label.strip()
    value = value.strip()
    value_confidence = label_item["confidence"]

    # A real label contains letters; this skips times such as 16:40
    if not any(character.isalpha() for character in label):
        continue

    # If the value is separate, look on the same line and just below
    if not value:
        right_values = []
        below_values = []
        label_height = label_item["y2"] - label_item["y1"]
        label_width = label_item["x2"] - label_item["x1"]
        label_center = (label_item["y1"] + label_item["y2"]) / 2

        for value_item in ocr_data:
            candidate_text = value_item["text"]
            candidate_label = candidate_text.split(":", maxsplit=1)[0]
            looks_like_another_label = (
                ":" in candidate_text
                and any(character.isalpha() for character in candidate_label)
            )

            if looks_like_another_label:
                continue

            value_center = (value_item["y1"] + value_item["y2"]) / 2
            is_same_page = value_item["page"] == label_item["page"]
            is_to_the_right = value_item["x1"] >= label_item["x2"]
            is_same_line = abs(value_center - label_center) <= label_height
            is_close_below = (
                label_item["y2"] <= value_item["y1"] <= label_item["y2"] + (2 * label_height)
                and abs(value_item["x1"] - label_item["x1"]) <= label_width
            )

            if not is_same_page:
                continue

            if is_to_the_right and is_same_line:
                distance = value_item["x1"] - label_item["x2"]
                right_values.append((distance, value_item))
            elif is_close_below:
                distance = value_item["y1"] - label_item["y2"]
                below_values.append((distance, value_item))

        nearest_value = None

        if right_values:
            nearest_value = min(right_values, key=lambda item: item[0])[1]
            clean_value = nearest_value["text"].strip().strip(".")

            # A short number may be an annotation, not the real value
            if clean_value.isdigit() and len(clean_value) <= 2 and below_values:
                nearest_value = min(below_values, key=lambda item: item[0])[1]
        elif below_values:
            nearest_value = min(below_values, key=lambda item: item[0])[1]

        if nearest_value is not None:
            value = nearest_value["text"]
            value_confidence = nearest_value["confidence"]

    if label and value:
        metadata_rows.append({
            "page": label_item["page"],
            "label": label,
            "value": value,
            "confidence": min(label_item["confidence"], value_confidence),
        })

metadata_table = pd.DataFrame(metadata_rows)
print("Metadata pairs found:", len(metadata_table))
metadata_table

Metadata pairs found: 16


,page,label,value,confidence
0,1,Report Date/Time,02/10/2014,0.9948
1,1,Name,"Doe, John Q.",0.9999
2,1,Age/Sex,73/M,1.0000
3,1,DOB,01/01/1941,0.9995
4,1,Patient ID,987654321,0.9978
5,1,Status,Routine,1.0000
6,1,Ordering Dr,"Smith, Peter MD",0.9997
7,1,Physician Copy for,"Smith, Jane MD",0.9810
8,1,SPEC #,223456,0.9666
9,1,Collection Date/Time,02/10/14,0.9998


## Part 3: Extract generic document structure and tables

Text OCR reads words separately. PP-StructureV3 adds layout information and groups content into blocks such as titles, paragraphs, and tables.

The first run downloads additional models, so it takes longer than later runs.

In [9]:
# Load the document structure model
structure_model = PPStructureV3(
    device="cpu",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    use_seal_recognition=False,
    use_formula_recognition=False,
    use_chart_recognition=False,
    use_table_recognition=True,
    enable_mkldnn=False,
)

print("Document structure model loaded")

Creating model: ('PP-DocBlockLayout', None, None)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Using official model (PP-DocBlockLayout), the model files will be automatically downloaded and saved in `C:\Users\user\.paddlex\official_models\PP-DocBlockLayout`.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('PP-DocLayout_plus-L', None, None)
Using official model (PP-DocLayout_plus-L), the model files will be automatically downloaded and saved in `C:\Users\user\.paddlex\official_models\PP-DocLayout_plus-L`.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('RT-DETR-L_wireless_table_cell_det', None, None)
Using official model (RT-DETR-L_wireless_table_cell_det), the model files will be automatically downloaded and saved in `C:\Users\user\.paddlex\official_models\RT-DETR-L_wireless_table_cell_det`.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Document structure model loaded


In [16]:
# Detect sections and tables in the report
structure_results = list(structure_model.predict(str(INPUT_PATH)))

print("Structured pages:", len(structure_results))

Structured pages: 1


In [11]:
# Create a small and easy structured result
structured_pages = []
structured_rows = []

for page_number, result in enumerate(structure_results, start=1):
    page_data = result.json

    if "res" in page_data:
        page_data = page_data["res"]

    page_blocks = []

    for block in page_data.get("parsing_res_list", []):
        simple_block = {
            "type": block.get("block_label", "unknown"),
            "content": block.get("block_content", ""),
            "box": block.get("block_bbox", []),
        }

        page_blocks.append(simple_block)
        structured_rows.append({
            "page": page_number,
            **simple_block,
        })

    structured_pages.append({
        "page": page_number,
        "blocks": page_blocks,
    })

structured_table = pd.DataFrame(structured_rows)
print("Document blocks found:", len(structured_table))
structured_table.head(30)

Document blocks found: 21


,page,type,content,box
0,1,paragraph_title,THE COMPLETE BLOOD COUNT SAMPLE REPORT,"[314, 109, 944, 131]"
1,1,text,Different laboratories generate reports that c...,"[93, 159, 1152, 272]"
2,1,table,<html><body><table><tbody><tr><td></td><td col...,"[70, 340, 1148, 1468]"
3,1,text,1.Name and address of the lab where the test w...,"[1247, 115, 1823, 209]"
4,1,text,2.Date this copy of the report was printed. Th...,"[1246, 223, 1826, 316]"
5,1,text,3.Patient name or identifier. Links results to...,"[1246, 331, 1825, 352]"
6,1,text,4.Patient identifier and identification number...,"[1246, 368, 1811, 413]"
7,1,text,5.Name of doctor. The lab will send the result...,"[1246, 427, 1824, 472]"
8,1,text,"6.Status of the test request, such as Routine ...","[1247, 488, 1816, 533]"
9,1,text,7.Unigue identification number(s). Number(s) a...,"[1247, 548, 1812, 593]"


In [12]:
# Count each type of document block
if structured_table.empty:
    print("No document blocks were found")
else:
    block_counts = structured_table["type"].value_counts()
    display(block_counts.rename_axis("type").to_frame("count"))

    table_blocks = structured_table[structured_table["type"] == "table"]
    print("Tables found:", len(table_blocks))

,count
type,
text,19
paragraph_title,1
table,1


Tables found: 1


In [13]:
# Convert detected HTML tables into pandas tables
extracted_tables = []

for block in structured_rows:
    if block["type"] != "table":
        continue

    try:
        tables_in_block = pd.read_html(StringIO(block["content"]))
        extracted_tables.extend(tables_in_block)
    except ValueError:
        continue

print("Pandas tables created:", len(extracted_tables))

for table_number, table in enumerate(extracted_tables, start=1):
    print(f"Table {table_number}")
    display(table)

Pandas tables created: 1
Table 1


,0,1,2,3,4,5,6,7
0,NaN,"University Medical Center, Dept. of Pathology","University Medical Center, Dept. of Pathology","University Medical Center, Dept. of Pathology","University Medical Center, Dept. of Pathology",Report Date/Time:,Report Date/Time:,NaN
1,"123 University Way, City, ST 12345",02/10/2014,02/10/2014,02/10/2014,16:40 2,16:40 2,NaN,NaN
2,3 4 5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Name:,"Doe, John Q.",Age/Sex:,Age/Sex:,Age/Sex:,73/M,73/M,73/M
4,Patient ID:,987654321,Status: 6,Status: 6,Status: 6,Routine,Routine,5
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,7,SPEC #:,223456,223456,223456,9.5 L**,02/10/14,14:30
7,NaN,NaN,Received Date/Time:,Received Date/Time:,Received Date/Time:,Received Date/Time:,Received Date/Time:,Received Date/Time:
8,8 9,SPECIMEN:,Whole blood,Whole blood,Whole blood,Whole blood,Whole blood,Whole blood
9,QUERIES:,[Comments and testing instructions],NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
# Save metadata, document blocks, tables, and Markdown
STRUCTURE_OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

with (STRUCTURE_OUTPUT_FOLDER / "structured_result.json").open("w", encoding="utf-8") as file:
    json.dump(structured_pages, file, indent=2, ensure_ascii=False)

structured_table.to_csv(
    STRUCTURE_OUTPUT_FOLDER / "document_blocks.csv",
    index=False,
)
metadata_table.to_csv(
    STRUCTURE_OUTPUT_FOLDER / "metadata_pairs.csv",
    index=False,
)

for table_number, table in enumerate(extracted_tables, start=1):
    table.to_csv(
        STRUCTURE_OUTPUT_FOLDER / f"table_{table_number}.csv",
        index=False,
    )

for page_number, result in enumerate(structure_results, start=1):
    result.save_to_json(
        str(STRUCTURE_OUTPUT_FOLDER / f"paddle_page_{page_number}.json")
    )
    result.save_to_markdown(
        str(STRUCTURE_OUTPUT_FOLDER / f"page_{page_number}.md")
    )

print("Structured files saved in:", STRUCTURE_OUTPUT_FOLDER.resolve())

Structured files saved in: C:\Users\user\Desktop\Paddle OCR\outputs\structure


In [15]:
# Display the first structured page inside JupyterLab
first_markdown_file = STRUCTURE_OUTPUT_FOLDER / "page_1.md"

if first_markdown_file.exists():
    markdown_text = first_markdown_file.read_text(encoding="utf-8")
    display(Markdown(markdown_text))
else:
    print("No Markdown file was created")

## THE COMPLETE BLOOD COUNT SAMPLE REPORT 

Different laboratories generate reports that can vary greatly in appearance and in the order and kind of information included. This is one example of what a lab report for a Complete Blood Count may look like.Names and places used have been made up for illustrative purposes only. The numbered key to the right explains a few of the report elements.




<div style="text-align: center;"><html><body><table border="1"><tbody><tr><td></td><td colspan="4">University Medical Center, Dept. of Pathology</td><td colspan="2">Report Date/Time:</td><td></td></tr><tr><td>123 University Way, City, ST 12345</td><td colspan="3">02/10/2014</td><td colspan="2">16:40 2</td><td></td></tr><tr><td>3 4 5</td><td colspan="3"></td><td></td><td></td><td></td><td></td></tr><tr><td>Name:</td><td>Doe, John Q.</td><td colspan="3">Age/Sex:</td><td colspan="3">73/M</td></tr><tr><td>Patient ID:</td><td>987654321</td><td colspan="3">Status: 6</td><td colspan="2">Routine</td><td>5</td></tr><tr><td></td><td></td><td colspan="3"></td><td colspan="2"></td><td></td></tr><tr><td>7</td><td>SPEC #:</td><td colspan="3">223456</td><td>9.5 L** </td><td>02/10/14</td><td>14:30</td></tr><tr><td></td><td></td><td colspan="6">Received Date/Time:</td></tr><tr><td>8 9</td><td>SPECIMEN:</td><td colspan="6">Whole blood</td></tr><tr><td>QUERIES:</td><td>[Comments and testing instructions]</td><td colspan="6"></td></tr><tr><td></td><td></td><td colspan="6"></td></tr><tr><td></td><td>15 17</td><td colspan="6"></td></tr><tr><td></td><td colspan="6">Test Normal Abnormal Flag Units Reference Range</td><td></td></tr><tr><td></td><td colspan="6"></td><td></td></tr><tr><td>White Blood Cell (WBC)</td><td colspan="2">6.9</td><td></td><td></td><td>K/mcL</td><td>4.8-10.8</td><td></td></tr><tr><td>Red Blood Cell (RBC)</td><td></td><td>1.8</td><td>L</td><td>M/mcL</td><td>4.7-6.1</td><td>Hemoglobin (HB/Hgb))</td><td></td></tr><tr><td>Hematocrit (HCT)</td><td></td><td>19.5</td><td>L**</td><td>%</td><td>42-52</td><td></td><td></td></tr><tr><td>Mean Cell Volume (MCV)</td><td></td><td>109.6</td><td>H</td><td>fL</td><td>80-100</td><td></td><td></td></tr><tr><td>Mean Cell Hemoglobin (MCH)</td><td></td><td>36.5</td><td>H</td><td>pg</td><td>27.0-32.0</td><td>Mean Cell Hb Conc (MCHC)</td><td>33.3</td></tr><tr><td>Red Cell Dist Width (RDW)</td><td></td><td>16.0</td><td>H</td><td>%</td><td>11.5-14.5</td><td></td><td></td></tr><tr><td>Platelet count</td><td>180</td><td></td><td></td><td>K/mcL</td><td>150-450</td><td></td><td></td></tr><tr><td>Mean Platelet Volume</td><td>7.9</td><td></td><td></td><td>fL</td><td>7.5-11.0</td><td></td><td>WBC Differential</td></tr><tr><td>Neutrophil (Neut)</td><td>50</td><td></td><td></td><td>%</td><td>33-73</td><td>Lymphocyte (Lymph)</td><td>36</td></tr><tr><td>Monocyte (Mono)</td><td>8</td><td></td><td></td><td>%</td><td>0-10</td><td></td><td></td></tr><tr><td>Eosinophil (Eos)</td><td>5</td><td></td><td></td><td>%</td><td>0-5</td><td></td><td></td></tr><tr><td colspan="8"></td></tr><tr><td>Basophil (Baso)</td><td>1</td><td colspan="2"></td><td></td><td>%</td><td>0-2</td><td>Neutrophil, Absolute</td></tr><tr><td>Lymphocyte, Absolute</td><td>2.5</td><td></td><td></td><td>K/mcL</td><td>1.0-4.8</td><td></td><td></td></tr><tr><td>Monocyte, Absolute</td><td>0.6</td><td></td><td></td><td>K/mcL</td><td>0-0.8</td><td></td><td></td></tr><tr><td>Eosinophil, Absolute</td><td>0.4</td><td></td><td></td><td>K/mcL</td><td>0-0.45</td><td>Basophil, Absolute</td><td>0.1</td></tr><tr><td>Flag Key: L= Abnormal Low, H= Abnormal High, **= critical value 18 Comment: **Hgb of 6.5 and Hct of 19.5 reported to Dr.J Smith at 15:20 on 2/10/14 by M. Petrs <div style="text-align: center;"><img src="imgs/img_in_image_box_973_1379_1024_1429.jpg" alt="Image" /></div></td><td>** END OF REPORT **</td><td></td><td></td><td></td><td></td><td></td><td></td></tr><tr><td></td><td></td><td></td><td></td><td></td><td></td><td></td><td></td></tr><tr><td></td><td></td><td></td><td></td><td></td><td></td><td></td><td></td></tr><tr><td></td><td></td><td></td><td></td><td></td><td></td><td></td><td></td></tr><tr><td></td><td></td><td></td><td></td><td></td><td></td><td></td><td></td></tr><tr><td></td><td></td><td></td><td></td><td></td><td></td><td></td><td></td></tr><tr><td></td><td></td><td></td><td></td><td></td><td></td><td></td><td></td></tr><tr><td colspan="8"></td></tr><tr><td colspan="8"></td></tr></tbody></table></body></html></div>


1.Name and address of the lab where the test was performed.Tests may be run in a physician office lab, a lab located in a clinic or hospital, and/or samples may be sent to a reference laboratory for analysis.



2.Date this copy of the report was printed. This date may be different than the date the results were generated, especially on cumulative reports (those that include results of several different tests run on different days).



3.Patient name or identifier. Links results to the correct person.

4.Patient identifier and identification number. Links results to the correct person.



5.Name of doctor. The lab will send the results to the doctor(s)or other healthcare practitioners listed.



6.Status of the test request, such as Routine or STAT (perform test as rapidly as possible).



7.Unigue identification number(s). Number(s) assigned to the sample(s) when it arrives at the laboratory.

8.Test requested is a CBC and WBC diferential.

9.Information about the person and blood sample. Any pertinent information regarding the patient's test preparation or the condition of specimen may be noted here.

10. The date and time of sample collection 

11. The date and time that the laboratory received the sample.

12. A listing of the individual items that are being evaluated.Test names may be abbreviated on lab reports. You can look for these test names or abbreviations in the pull-down menu on the home page of this site or type the name into the search box to find information on specific tests.

13.A listing of the CBC and differential results that are normal.

14.A listing of the CBC and differential results that are abnormal.

15.An 'H' in this column may mean that the result is higher than the reference range. 'L' may mean 'low.' Either represents a result outside the reference range/value.



16. Units of measurement (for quantitative results). The units of measurement that labs use to report your results can vary from lab to lab. Regardless of the units that the lab uses, your results will be interpreted in relation to the reference ranges supplied by the laboratory 



17. Reference intervals (or reference ranges). These are the ranges in which "normal" values are expected to fall The ranges that appear on your report are validated and supplied by the laboratory that performed your test.

18. Critical results are dangerously abnormal results that must be reported immediately to the responsible person, such as the ordering physician. The laboratory will often draw attention to such results with an asterisk (*) or something similar and will usually note on the report the date and time the responsible person was notified.



## What the structured output means

- `type` tells you whether a block is a title, normal text, a table, or another layout element.
- `content` contains the recognized text or table content.
- `box` contains the block position on the page.

The metadata cell uses the common `Label: Value` pattern, and table columns keep the headings found in the report. These are generic layout rules. They do **not** guarantee that every laboratory format will become the same medical schema. Standard fields such as `patient_name`, `test_name`, `value`, `unit`, and `reference_range` need a later semantic mapping and validation step.